# NB01 — Phase 0: train the four pilot models

        **4 runs · ~12 GPU-hours · shardable across up to 4 accounts**


> **New here?** Read `05_PLAIN_ENGLISH_GUIDE.md` first — it explains what this
> project is measuring and why, without jargon. This notebook assumes you have.


        ## What we're doing and why

        Before spending ~1,200 GPU-hours, we spend 12 to find out whether the
        thing we want to measure is even measurable.

        Four models:

        | Model | Seed | Why |
        |---|---|---|
        | resnet32x4 | 1 | reference model A |
        | resnet32x4 | 2 | **the same model, different random start** |
        | wrn-40-2 | 1 | a different architecture, B |
        | wrn-40-2 | 2 | **same again, different random start** |

        The duplicate seeds are not redundancy. They answer: *when we train the
        exact same thing twice, do the two copies agree about which images are
        hard?*

        That agreement is our **noise ceiling** — the reference point that makes
        every later number interpretable. If two copies of the same model only
        agree 60% of the time, then a ResNet and a ViT agreeing 55% is actually
        near-perfect. Without the ceiling, 55% is just a number.

        ## What to expect

        These are standard, published architectures on a standard recipe, so we
        know what accuracy they should reach: **resnet32x4 ≈ 79.4%**,
        **wrn-40-2 ≈ 75.6%**.

        The notebook checks this and warns loudly if a run lands more than 1
        point low. That matters because measuring "how much compute does this
        image need" on a badly-trained model gives meaningless answers — and a
        badly-trained model is otherwise easy to miss.

        ## Timing

        ~3 hours per model. With `NUM_WORKERS = 4`, one per account, about
        3 hours total. With 1 worker, about 12 hours across two sessions
        (it pauses and resumes automatically at the 8.5-hour mark).

In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# This writes two Python files into the session and imports them. Nothing here
# touches the GPU or the network beyond installing three small packages.
#
#   msc_lib   c36dccac582f   the pipeline: HuggingFace sync, model zoo,
#                              measurement, training, the method
#   msc_core  6abdba4ff104   the reference maths: the MSC definition and
#                              every statistic in the paper
#
# Both are generated from KD/src by build_notebooks.py. Editing them HERE does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle images already ship torch, pandas and sklearn. These three vary by
# image version, so we check rather than assume.
#   pyarrow  writes the per-image measurement tables (Parquet)
#   pynvml   reads GPU power/temperature/utilisation directly
#   fvcore   counts FLOPs, which is how compute cost is defined
for _pkg in ('pyarrow', 'pynvml', 'fvcore', 'psutil'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg,
                        '--break-system-packages'], check=False)

_LIB = (
    '...',
    'bGluZSBjaGVja3MiKQo=',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]

import msc_lib as msc
import msc_core

print(f'[BOOT] msc_lib v{msc.__version__} ready  (torch available: {msc._TORCH_OK})')
print(f'[BOOT] artifact space: {msc.WORK_ROOT}   scratch space: {msc.SCRATCH_ROOT}')

## Step 1 — Session (set your WORKER_ID)

In [ ]:
# === Who am I? =============================================================
#
# THIS NOTEBOOK: 4 run(s), ~10 GPU-hours total.
# At NUM_WORKERS = 1 that is all of it on this account.
# Raise NUM_WORKERS and run the same notebook on each account
# with a different WORKER_ID to divide it.
#
# ACCOUNT   labels this Kaggle account in the run log. Two accounts calling
#           themselves the same thing makes the log useless.
# WORKER_ID splits the work. Every account runs THE SAME notebook; the only
#           thing that differs is this number. Account 1 -> 0, account 2 -> 1,
#           and so on. Each account then works out, by pure arithmetic, which
#           jobs belong to it -- no communication needed, no chance of two
#           accounts training the same model, no chance of a job being missed.
#
# DEFAULT IS 1: this account does everything in this notebook. That is the
# simplest thing that works. Change it only when you actually have several
# accounts running at once.
# Phase 0 is 4 runs, so up to 4 accounts can help.
ACCOUNT     = 'acct1'      # <<< CHANGE ME
NUM_WORKERS = 1          # <<< how many accounts you are running in parallel
WORKER_ID   = 0            # <<< CHANGE ME: 0, 1, 2, ... up to NUM_WORKERS-1

sess = msc.Session(account=ACCOUNT, phase='p0', dataset='cifar100',
                   worker_id=WORKER_ID, num_workers=NUM_WORKERS,
                   shard_mode='cost',        # balance GPU-hours, not job counts
                   enable_hf=True,
                   session_limit_h=8.5,      # push + pause before Kaggle kills us
                   commits_per_hour_limit=20,# 6 accounts x 20 = 120 < HF's 128/hr
                   batch_interval_sec=1800)  # the 30-minute push policy

## Step 2 — Dataset

In [ ]:
# === Get CIFAR-100 =========================================================
# Looks in this order: attached Kaggle dataset (instant) -> earlier extraction
# -> Kaggle CLI download -> torchvision as a last resort.
# Everything lands in /kaggle/temp (~1 TB scratch), never in /kaggle/working
# (20 GB, and that is the space your results need).
DATA_ROOT = sess.prepare_data()
print('dataset:', DATA_ROOT)

## Step 3 — Catch up with prior progress

In [ ]:
# === Catch up with what has already been done ==============================
# Downloads only what this notebook needs -- never the whole repo, which would
# fill the 20 GB disk instantly.
#
# It also rebuilds the progress record from the actual training logs instead of
# trusting the status file. If a session died between writing a log and pushing
# its status, those two disagree, and the log is the one that tells the truth.
# Runs marked "finished" that clearly are not get reset so they resume.
sess.sync_state(verbose=True)
sess.status()

## Step 4 — Train

The cell below plans this worker's share, then trains it.

**You can stop this at any time.** Press the stop button, close the tab,
or let Kaggle time out — everything is pushed to HuggingFace first. To
continue, open a fresh session and run all cells again; it resumes from
the exact epoch it stopped at.

In [ ]:
EPOCHS = 240        # the standard published recipe. Do not shorten for Phase 0 --
                    # under-trained models give meaningless measurements.

cfgs = [sess.config(a, seed=s, num_epochs=EPOCHS)
        for a in ('resnet32x4', 'wrn_40_2') for s in (1, 2)]
run_ids = [c['run_id'] for c in cfgs]

est = msc.estimate_phase(run_ids, num_workers=NUM_WORKERS)
print(f"{est['n_runs']} runs, ~{est['total_gpu_hours']:.1f} GPU-hours total")
print(f"at NUM_WORKERS={NUM_WORKERS}: ~{est['wall_clock_hours']:.1f} h wall-clock, "
      f"{est['sessions_needed']} session(s)")
for r in run_ids:
    print(f"  {r:34s} ~{msc.estimate_run_hours(r):.2f} h")
print()

summaries = sess.run_all(cfgs, title='Phase 0 training')

import pandas as pd
pd.DataFrame([{k: s.get(k) for k in
               ('run_id', 'best_accuracy', 'reference_accuracy',
                'accuracy_gap_vs_reference', 'recipe_ok', 'num_epochs_run',
                'total_time_sec', 'total_energy_kwh')}
              for s in summaries if s.get('status') == 'completed'])

## Step 5 — Did the training actually work?

Compares against the published numbers. Anything more than 1 point below
means the recipe is wrong, and everything measured from that model would
be worthless.

In [ ]:
import pandas as pd
rows = []
for rid, st in sess.registry.latest().items():
    if st.get('state') != 'completed' or not rid.startswith('p0-'):
        continue
    ref, acc = msc.REFERENCE_ACC.get(st.get('arch')), st.get('best_accuracy')
    if ref and acc:
        rows.append({'run_id': rid, 'accuracy_%': round(acc * 100, 2),
                     'published_%': ref, 'gap': round(ref - acc * 100, 2),
                     'ok': (ref - acc * 100) <= 1.0})
audit = pd.DataFrame(rows)
if len(audit):
    display(audit)
if len(audit) and not audit.ok.all():
    print('\nSOME RUNS ARE UNDER-TRAINED. Fix the recipe before NB02.')
elif len(audit):
    print('\nAll runs match their published accuracy. Proceed to NB02.')
else:
    print('\nNo completed Phase 0 runs yet on this account (others may have them).')

## Step 6 — Audit HuggingFace

Lists what is **actually** in both repositories, per run, so you can see
at a glance whether anything is half-pushed or missing.

Two things to look at:

- **`ledger shards`** should equal the number of worker sessions that
  have run. If it says 0, you're on an older build of the library —
  re-upload the notebooks.
- **`NOT STARTED`** lists runs nobody has picked up. With
  `NUM_WORKERS = 4` and 4 runs, every worker owns exactly one, so a run
  appearing here means that worker's session hasn't been started.
- **`FOREIGN DATA`** flags runs that don't match any architecture in the
  current zoo — usually leftovers from an earlier version of the
  project. Harmless to the analysis, but worth clearing out.

In [ ]:
audit = sess.audit_repos(expected_run_ids=[c['run_id'] for c in cfgs])

## Step 7 — Finish

In [ ]:
# === Push everything and stop ==============================================
# Blocks until HuggingFace confirms. Safe to re-run.
sess.finish()

# D-19: draining the upload queue is NOT the same as the files being on
# HuggingFace, and "[SESSION] done" reads like a confirmation it is not.
# Ask the repository before you close this tab.
#
# D-20: three states, not two. FINISHED and RESUMABLE are both safe -- a run
# paused at epoch 120 whose ckpt_last.pt is on HF loses nothing when you close
# the tab. Only AT RISK (no summary.json AND no checkpoint) needs action.
try:
    _ids = [c['run_id'] for c in (all_cfgs if 'all_cfgs' in dir() else cfgs)]
except NameError:
    _ids = []
if _ids:
    sess.confirm_on_hf(_ids)

---
**Next: NB02** — measure how much compute each image needs.